# Сравнение ORB VO, учебной прямой VO и двух ГНСС

Протокол: исходные ROS timestamps → ближайшая временная ассоциация → один Umeyama Sim(3) на всю сопоставленную траекторию → ATE, трансляционная RPE на 1 с и локальный масштаб. Сначала прямой метод выравнивается на ORB в произвольных единицах, затем каждый алгоритм отдельно выравнивается на каждую ГНСС в метрах.

ГНСС — референс положения **антенны**, а не камеры. Плечо антенна–камера не компенсируется, RTK-статус читается из GGA quality. Для общего рисунка обе VO приведены к ENU ГНСС1; независимое выравнивание на ГНСС2 используется только в отдельных метриках. Оба ноутбука VO учебные, поэтому это сравнение реализаций на данных, а не бенчмарк оригинальных ORB-SLAM2 и DSO.

In [ ]:
from pathlib import Path
import csv
import json

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == 'scripts' else Path.cwd()
RESULTS_DIR = ROOT / 'results' / 'metrics'
FIGURES_DIR = RESULTS_DIR / 'figures'
VO_DIR = ROOT / 'results' / 'VO'
GNSS_DIR = ROOT / 'results' / 'GNSS'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TRAJECTORY_PATHS = {'ORB': VO_DIR / 'ORB' / 'orb_tum.txt', 'DSO': VO_DIR / 'DSO' / 'dso_tum.txt'}
GNSS_PATHS = {'ГНСС 1': GNSS_DIR / 'gnss_one.csv', 'ГНСС 2': GNSS_DIR / 'gnss_two.csv'}
ASSOCIATION_TOLERANCE = 0.06
RPE_DELTA_SECONDS = 1.0
RPE_TIME_TOLERANCE = 0.15

# Единое оформление всех рисунков для отчёта.
plt.rcParams['figure.figsize'] = (16, 8)
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman']
plt.rcParams['font.size'] = 18
for path in [*TRAJECTORY_PATHS.values(), *GNSS_PATHS.values()]:
    if not path.is_file() or path.stat().st_size == 0:
        raise FileNotFoundError(f'Required result is missing or empty: {path}')

## 1. Загрузка и временная ассоциация

Обе ГНСС уже переведены в общий ENU с началом в самой ранней валидной GGA-точке. Допуск 60 мс соответствует примерно половине периода 10 Гц плюс небольшой запас. Интерполяция ГНСС не выполняется: каждой VO-позе сопоставляется ближайшее реальное измерение приёмника.

In [2]:
def load_tum(path):
    data = np.loadtxt(path, ndmin=2)
    if data.shape[1] != 8 or not np.isfinite(data).all():
        raise ValueError(f'{path}: expected finite 8-column TUM data')
    norms = np.linalg.norm(data[:, 4:8], axis=1)
    if np.any(np.abs(norms - 1) > 1e-4):
        raise ValueError(f'{path}: invalid quaternion norm')
    order = np.argsort(data[:, 0])
    return data[order, 0], data[order, 1:4]

def load_gnss(path):
    data = np.genfromtxt(path, delimiter=',', names=True)
    order = np.argsort(data['timestamp'])
    return (data['timestamp'][order], np.column_stack((data['east'][order], data['north'][order], data['up'][order])),
            data['quality'][order].astype(int))

def associate(source_time, source_xyz, reference_time, reference_xyz, tolerance=ASSOCIATION_TOLERANCE):
    source_indices, reference_indices = [], []
    for index, timestamp in enumerate(source_time):
        insertion = np.searchsorted(reference_time, timestamp)
        candidates = [i for i in (insertion - 1, insertion) if 0 <= i < len(reference_time)]
        if not candidates:
            continue
        nearest = min(candidates, key=lambda i: abs(reference_time[i] - timestamp))
        if abs(reference_time[nearest] - timestamp) <= tolerance:
            source_indices.append(index); reference_indices.append(nearest)
    if len(source_indices) < 3:
        raise RuntimeError('Недостаточно временно сопоставленных поз')
    source_indices, reference_indices = np.array(source_indices), np.array(reference_indices)
    return (source_time[source_indices], source_xyz[source_indices],
            reference_time[reference_indices], reference_xyz[reference_indices],
            reference_time[reference_indices] - source_time[source_indices])

trajectories = {name: load_tum(path) for name, path in TRAJECTORY_PATHS.items()}
gnss = {name: load_gnss(path) for name, path in GNSS_PATHS.items()}
camera_times = np.loadtxt(ROOT / 'calib' / 'times.txt')
for name, (time, xyz) in trajectories.items():
    print(f'{name}: {len(time)} poses, {time[-1] - time[0]:.3f} s, finite={np.isfinite(xyz).all()}')
for name, (time, xyz, quality) in gnss.items():
    camera_count = np.sum((time >= camera_times[0] - ASSOCIATION_TOLERANCE) &
                          (time <= camera_times[-1] + ASSOCIATION_TOLERANCE))
    print(f'{name}: {camera_count} fixes in camera interval; GGA quality={dict(zip(*np.unique(quality, return_counts=True)))}')

ORB: 589 poses, 59.899 s, finite=True
DSO: 590 poses, 59.899 s, finite=True
GNSS 1: 600 fixes in camera interval; GGA quality={np.int64(1): np.int64(22525)}
GNSS 2: 600 fixes in camera interval; GGA quality={np.int64(1): np.int64(22507)}


## 2. Sim(3), ATE, RPE и масштаб

Один глобальный Sim(3) одновременно оценивает поворот, перевод и единый масштаб всей VO-траектории. Локальный подбор масштаба запрещён, иначе ошибка дрейфа была бы скрыта.

- **ATE** — расстояние между выровненной позой и синхронным ГНСС-положением; приводим RMSE, среднее, медиану, стандартное отклонение, 95-й процентиль и максимум.
- **RPE за 1 с** — ошибка относительного приращения; приводим RMSE, медиану, максимум и медианную относительную ошибку в процентах пройденного ГНСС-пути.
- **Local scale** — отношение длины приращения VO к приращению ГНСС. После идеального глобального выравнивания медиана близка к 1, а узкий интервал p10–p90 означает стабильный масштаб.

RPE и масштаб вычисляются только там, где найдены два момента с интервалом, близким к 1 с. Относительная RPE и масштаб дополнительно требуют ГНСС-приращения более 0,1 м, чтобы не делить на шум почти неподвижного приёмника.

In [ ]:
def align_sim3(source_xyz, reference_xyz):
    # Umeyama: центрирование → SVD → поворот без отражения → единый масштаб и перевод.
    source_mean, reference_mean = source_xyz.mean(axis=0), reference_xyz.mean(axis=0)
    source_centered = source_xyz - source_mean
    reference_centered = reference_xyz - reference_mean
    u, singular, vt = np.linalg.svd(reference_centered.T @ source_centered / len(source_xyz))
    correction = np.eye(3)
    if np.linalg.det(u @ vt) < 0:
        correction[-1, -1] = -1
    rotation = u @ correction @ vt
    scale = len(source_xyz) * np.sum(singular * np.diag(correction)) / np.sum(source_centered ** 2)
    translation = reference_mean - scale * rotation @ source_mean
    aligned = (scale * rotation @ source_xyz.T).T + translation
    return aligned, float(scale), rotation, translation

def one_second_pairs(source_time):
    # Для каждой позы ищем ближайшую позу через 1 с с допуском 0,15 с.
    pairs = []
    for index, timestamp in enumerate(source_time):
        target = timestamp + RPE_DELTA_SECONDS
        insertion = np.searchsorted(source_time, target)
        candidates = [i for i in (insertion - 1, insertion) if 0 <= i < len(source_time)]
        if not candidates:
            continue
        nearest = min(candidates, key=lambda i: abs(source_time[i] - target))
        if nearest > index and abs(source_time[nearest] - target) <= RPE_TIME_TOLERANCE:
            pairs.append((index, nearest))
    return pairs

def calculate_metrics(aligned, reference, source_time):
    ate = np.linalg.norm(aligned - reference, axis=1)
    pairs = one_second_pairs(source_time)
    if pairs:
        first, second = np.array(pairs).T
        source_steps = aligned[second] - aligned[first]
        reference_steps = reference[second] - reference[first]
        rpe = np.linalg.norm(source_steps - reference_steps, axis=1)
        reference_length = np.linalg.norm(reference_steps, axis=1)
        local_scale = np.linalg.norm(source_steps, axis=1) / np.maximum(reference_length, 1e-9)
        valid_scale = reference_length > 0.1
        scale_values = local_scale[valid_scale]
        relative_rpe = 100.0 * rpe[valid_scale] / reference_length[valid_scale]
    else:
        rpe, scale_values, relative_rpe = np.array([]), np.array([]), np.array([])
    return {
        'ATE_RMSE': float(np.sqrt(np.mean(ate ** 2))),
        'ATE_mean': float(np.mean(ate)),
        'ATE_median': float(np.median(ate)),
        'ATE_std': float(np.std(ate)),
        'ATE_p95': float(np.percentile(ate, 95)),
        'ATE_max': float(np.max(ate)),
        'RPE_RMSE_1s': float(np.sqrt(np.mean(rpe ** 2))) if len(rpe) else np.nan,
        'RPE_median_1s': float(np.median(rpe)) if len(rpe) else np.nan,
        'RPE_max_1s': float(np.max(rpe)) if len(rpe) else np.nan,
        'RPE_relative_median_percent': float(np.median(relative_rpe)) if len(relative_rpe) else np.nan,
        'local_scale_median': float(np.median(scale_values)) if len(scale_values) else np.nan,
        'local_scale_p10': float(np.percentile(scale_values, 10)) if len(scale_values) else np.nan,
        'local_scale_p90': float(np.percentile(scale_values, 90)) if len(scale_values) else np.nan,
        'RPE_pairs': len(pairs),
    }, ate, pairs

def evaluate(source, reference):
    source_time, source_xyz, reference_time, reference_xyz, time_offset = associate(
        source[0], source[1], reference[0], reference[1])
    aligned, scale, rotation, translation = align_sim3(source_xyz, reference_xyz)
    metrics, ate, pairs = calculate_metrics(aligned, reference_xyz, source_time)
    metrics.update(global_sim3_scale=scale, associated_poses=len(source_time),
                   association_dt_median_ms=float(np.median(np.abs(time_offset)) * 1000),
                   association_dt_max_ms=float(np.max(np.abs(time_offset)) * 1000))
    return {'metrics': metrics, 'time': source_time, 'aligned': aligned, 'reference': reference_xyz,
            'ate': ate, 'rpe_pairs': pairs, 'scale': scale, 'rotation': rotation, 'translation': translation}

## 3. ORB ↔ прямой метод

Прямая VO выравнивается на ORB одним Sim(3). Здесь ATE/RPE остаются в произвольных единицах ORB: сравнение показывает согласованность формы и локальных приращений двух VO, но не метрическую точность. Оно не заменяет независимую оценку относительно ГНСС.

In [ ]:
orb_dso = evaluate(trajectories['DSO'], trajectories['ORB'])
print('Прямая VO -> ORB:', json.dumps(orb_dso['metrics'], indent=2))
figure, axes = plt.subplots(1, 2, figsize=(17, 8))
axes[0].plot(orb_dso['reference'][:, 0], orb_dso['reference'][:, 2], label='ORB: опорная траектория', color='tab:blue')
axes[0].plot(orb_dso['aligned'][:, 0], orb_dso['aligned'][:, 2], label='Прямая VO после Sim(3)', color='tab:purple', alpha=0.8)
axes[0].set(xlabel='X локальной карты ORB, усл. ед.', ylabel='Z локальной карты ORB, усл. ед.',
            title='Согласование прямой VO с ORB одним Sim(3)')
axes[0].axis('equal')
axes[0].grid(alpha=0.25)
axes[0].legend()
axes[1].plot(orb_dso['time'] - camera_times[0], orb_dso['ate'], color='tab:purple')
axes[1].set(xlabel='Время от начала записи камеры, с', ylabel='Ошибка ATE, усл. ед. ORB',
            title='Расхождение ORB и прямой VO')
axes[1].grid(alpha=0.25)
figure.tight_layout()
figure.savefig(FIGURES_DIR / 'orb_dso_comparison.png', dpi=200, bbox_inches='tight')

## 4. ORB и прямая VO относительно двух ГНСС

Каждое сравнение получает собственный Sim(3), потому что монокулярный масштаб и система координат неизвестны. Одинаковые алгоритм, допуски и формулы применяются к обоим референсам. Метрики относительно ГНСС2 не используются для общего графика: там обе VO выровнены на ГНСС1.

Важно: референс описывает антенну, а не оптический центр камеры. Неучтённое плечо и рысканье самосвала входят в итоговую ошибку, поэтому большие ATE нельзя интерпретировать как чистую ошибку алгоритма.

In [5]:
evaluations = {}
metric_rows = []
for gnss_name, reference in gnss.items():
    for algorithm_name, trajectory in trajectories.items():
        result = evaluate(trajectory, reference)
        key = f'{algorithm_name} vs {gnss_name}'
        evaluations[key] = result
        metrics = result['metrics']
        metric_rows.append({'comparison': key, 'units': 'm', **metrics})
        aligned_path = RESULTS_DIR / f'{algorithm_name.lower()}_{gnss_name.lower().replace(" ", "_")}_aligned.csv'
        with aligned_path.open('w', newline='', encoding='utf-8') as stream:
            writer = csv.writer(stream)
            writer.writerow(['timestamp', 'aligned_east', 'aligned_north', 'aligned_up',
                             'gnss_east', 'gnss_north', 'gnss_up', 'ATE_m'])
            for values in zip(result['time'], result['aligned'], result['reference'], result['ate']):
                timestamp, aligned, reference_xyz, ate = values
                writer.writerow([timestamp, *aligned, *reference_xyz, ate])
        print(key, json.dumps(metrics, indent=2))

orb_dso_row = {'comparison': 'DSO vs ORB', 'units': 'ORB arbitrary units', **orb_dso['metrics']}
with (RESULTS_DIR / 'comparison_metrics.csv').open('w', newline='', encoding='utf-8') as stream:
    fieldnames = list(orb_dso_row)
    writer = csv.DictWriter(stream, fieldnames=fieldnames)
    writer.writeheader(); writer.writerow(orb_dso_row); writer.writerows(metric_rows)
(RESULTS_DIR / 'comparison_metrics.json').write_text(json.dumps(
    {'DSO vs ORB': orb_dso['metrics'], **{key: value['metrics'] for key, value in evaluations.items()}},
    ensure_ascii=False, indent=2), encoding='utf-8')

ORB vs GNSS 1 {
  "ATE_RMSE": 133.30840039129077,
  "ATE_median": 108.18571819277568,
  "ATE_max": 290.6130568157395,
  "RPE_RMSE_1s": 37.31668849427739,
  "local_scale_median": 1.7932151058048795,
  "local_scale_p10": 0.6096403704571977,
  "local_scale_p90": 4.411971454633761,
  "RPE_pairs": 561,
  "global_sim3_scale": 29.503169844471948,
  "associated_poses": 578,
  "association_dt_median_ms": 20.978689193725586,
  "association_dt_max_ms": 50.26578903198242
}
DSO vs GNSS 1 {
  "ATE_RMSE": 63.2939633083178,
  "ATE_median": 32.10069158892688,
  "ATE_max": 223.49669924799284,
  "RPE_RMSE_1s": 28.989744065835065,
  "local_scale_median": 0.9412450198054391,
  "local_scale_p10": 0.3640395466867981,
  "local_scale_p90": 3.5450836337276677,
  "RPE_pairs": 562,
  "global_sim3_scale": 249.2003337925686,
  "associated_poses": 579,
  "association_dt_median_ms": 20.978689193725586,
  "association_dt_max_ms": 50.26578903198242
}
ORB vs GNSS 2 {
  "ATE_RMSE": 134.3395605127134,
  "ATE_median": 109.

## 5. Все траектории на одном графике

На первом рисунке показаны исходные ГНСС1/ГНСС2 в общем ENU и обе VO после **их отдельного выравнивания на ГНСС1**. Так сохраняется одна система координат. Если встречаются GGA quality 4 (RTK fixed) или 5 (RTK float), соответствующие ГНСС-участки дополнительно выделяются цветом; при quality 1 RTK-поправок нет.

Далее оставлены отдельные сравнения с каждым приёмником и ошибка ATE во времени. Разные Sim(3) на отдельных панелях не смешиваются на одном графике.

In [ ]:
# Общий отчётный график: общая система ENU ГНСС1 для всех четырёх траекторий.
figure, axis = plt.subplots(figsize=(14, 10))
# Сравнение относится к минуте камеры, поэтому обе ГНСС обрезаем по тому же интервалу.
gnss_window = {}
for gnss_name in GNSS_PATHS:
    gnss_time, gnss_xyz, quality = gnss[gnss_name]
    in_camera_interval = ((gnss_time >= camera_times[0] - ASSOCIATION_TOLERANCE) &
                          (gnss_time <= camera_times[-1] + ASSOCIATION_TOLERANCE))
    gnss_window[gnss_name] = (gnss_time[in_camera_interval], gnss_xyz[in_camera_interval],
                              quality[in_camera_interval])
axis.plot(gnss_window['ГНСС 1'][1][:, 0], gnss_window['ГНСС 1'][1][:, 1], 'k--', linewidth=2.0, label='ГНСС 1')
axis.plot(gnss_window['ГНСС 2'][1][:, 0], gnss_window['ГНСС 2'][1][:, 1], color='tab:green', linestyle='--',
          linewidth=1.8, alpha=0.85, label='ГНСС 2')
for gnss_name, color in [('ГНСС 1', 'tab:orange'), ('ГНСС 2', 'tab:cyan')]:
    gnss_time, gnss_xyz, quality = gnss_window[gnss_name]
    rtk = np.isin(quality, [4, 5])
    if np.any(rtk):
        axis.scatter(gnss_xyz[rtk, 0], gnss_xyz[rtk, 1], s=16, color=color,
                     label=f'{gnss_name}: участки с RTK', zorder=4)
for algorithm_name, color in [('ORB', 'tab:blue'), ('DSO', 'tab:purple')]:
    result = evaluations[f'{algorithm_name} vs ГНСС 1']
    axis.plot(result['aligned'][:, 0], result['aligned'][:, 1], color=color, linewidth=1.5,
              label=f'{algorithm_name}: {result["time"][-1] - result["time"][0]:.1f} с; '
              f'ATE RMSE {result["metrics"]["ATE_RMSE"]:.2f} м')
axis.scatter(0, 0, color='red', s=70, label='Начало ENU', zorder=5)
axis.set(xlabel='Восток ENU, м', ylabel='Север ENU, м',
         title='Все траектории за интервал камеры в системе ГНСС 1')
axis.axis('equal')
axis.grid(alpha=0.25)
axis.legend()
figure.tight_layout()
figure.savefig(FIGURES_DIR / 'all_trajectories.png', dpi=200, bbox_inches='tight')

# Отдельные панели сохраняют независимую оценку каждого приёмника.
figure, axes = plt.subplots(1, 2, figsize=(18, 9))
for axis, gnss_name in zip(axes, GNSS_PATHS):
    for algorithm_name, color in [('ORB', 'tab:blue'), ('DSO', 'tab:purple')]:
        result = evaluations[f'{algorithm_name} vs {gnss_name}']
        axis.plot(result['reference'][:, 0], result['reference'][:, 1], 'k--', linewidth=2,
                  label=gnss_name if algorithm_name == 'ORB' else None)
        axis.plot(result['aligned'][:, 0], result['aligned'][:, 1], color=color, linewidth=1.2,
                  label=f'{algorithm_name}: ATE {result["metrics"]["ATE_RMSE"]:.2f} м')
    axis.set(xlabel='Восток ENU, м', ylabel='Север ENU, м',
             title=f'Траектории VO после выравнивания на {gnss_name}')
    axis.axis('equal')
    axis.grid(alpha=0.25)
    axis.legend()
figure.tight_layout()
figure.savefig(FIGURES_DIR / 'gnss_vo_comparison.png', dpi=200, bbox_inches='tight')

figure, axes = plt.subplots(2, 2, figsize=(17, 11), sharex=True)
for axis, (key, result) in zip(axes.ravel(), evaluations.items()):
    axis.plot(result['time'] - camera_times[0], result['ate'],
              color='tab:blue' if key.startswith('ORB') else 'tab:purple')
    russian_key = key.replace('GNSS', 'ГНСС')
    axis.set(title=f'Ошибка: {russian_key}', ylabel='ATE, м')
    axis.grid(alpha=0.25)
for axis in axes[-1]:
    axis.set_xlabel('Время от начала записи камеры, с')
figure.tight_layout()
figure.savefig(FIGURES_DIR / 'trajectory_ate.png', dpi=200, bbox_inches='tight')

## 6. Сводные метрики и внутренняя диагностика

ATE/RPE сравниваем вместе с полнотой трекинга: алгоритм с малой ошибкой на коротком фрагменте не обязательно лучше метода, сохранившего весь проезд. Внутренние inliers и фотометрическая невязка объясняют поведение трекинга, но не заменяют внешнюю ГНСС-оценку.

In [ ]:
orb_summary = json.loads((VO_DIR / 'ORB' / 'orb_summary.json').read_text(encoding='utf-8'))
dso_summary = json.loads((VO_DIR / 'DSO' / 'dso_summary.json').read_text(encoding='utf-8'))
print(f'Полнота трекинга: ORB={orb_summary["completeness"]:.2%}, прямая VO={dso_summary["completeness"]:.2%}')
print(f'Среднее время кадра: ORB={orb_summary["runtime_mean_s"] * 1000:.1f} мс, прямая VO={dso_summary["runtime_mean_s"] * 1000:.1f} мс')
print(f'95-й процентиль времени: ORB={orb_summary["runtime_p95_s"] * 1000:.1f} мс, прямая VO={dso_summary["runtime_p95_s"] * 1000:.1f} мс')
labels = []
for key in evaluations:
    coverage = orb_summary['completeness'] if key.startswith('ORB') else dso_summary['completeness']
    labels.append(f'{key.replace("GNSS", "ГНСС")}\nполнота {coverage:.1%}')
ate_values = [evaluations[key]['metrics']['ATE_RMSE'] for key in evaluations]
rpe_values = [evaluations[key]['metrics']['RPE_RMSE_1s'] for key in evaluations]
figure, axes = plt.subplots(1, 2, figsize=(18, 8))
x = np.arange(len(labels))
colors = ['tab:blue' if key.startswith('ORB') else 'tab:purple' for key in evaluations]
axes[0].bar(x, ate_values, color=colors)
axes[0].set_ylabel('ATE RMSE, м')
axes[0].set_title('Глобальная ошибка после одного Sim(3)')
axes[1].bar(x, rpe_values, color=colors)
axes[1].set_ylabel('Трансляционная RPE RMSE за 1 с, м')
axes[1].set_title('Локальный дрейф')
for axis in axes:
    axis.set_xticks(x, labels, rotation=25, ha='right')
    axis.grid(axis='y', alpha=0.25)
figure.tight_layout()
figure.savefig(FIGURES_DIR / 'metrics_summary.png', dpi=200, bbox_inches='tight')

# Таблица в stdout и CSV содержит больше метрик, чем можно читаемо разместить на столбцах.
for gnss_name in GNSS_PATHS:
    candidates = {name: evaluations[f'{name} vs {gnss_name}']['metrics']['ATE_RMSE'] for name in trajectories}
    best = min(candidates, key=candidates.get)
    other = next(name for name in candidates if name != best)
    print(f'По ATE RMSE относительно {gnss_name} лучше {best}: {candidates[best]:.3f} м; '
          f'{other}: {candidates[other]:.3f} м')
    for name in trajectories:
        metrics = evaluations[f'{name} vs {gnss_name}']['metrics']
        print(f'  {name}: RPE median={metrics["RPE_median_1s"]:.3f} м/с, '
              f'относительная RPE={metrics["RPE_relative_median_percent"]:.2f}%, '
              f'ATE p95={metrics["ATE_p95"]:.3f} м')
if min(orb_summary['completeness'], dso_summary['completeness']) < 0.95:
    print('ВЫВОД: формально меньший ATE ORB получен только на коротком фрагменте; '
          'по совокупности точности и полноты нельзя объявить ORB лучшим алгоритмом.')


## 7. Как интерпретировать результат

- Меньшие ATE и RPE лучше, но только вместе с высокой полнотой трекинга и достаточным числом временных ассоциаций.
- Большие `ATE_p95/ATE_max` показывают участки сильного расхождения даже при умеренной медиане.
- `local_scale_p10/p90`, далёкие от 1, показывают нестабильность масштаба после глобального выравнивания; относительная RPE позволяет сравнивать локальный дрейф с пройденным путём.
- Различие результатов между ГНСС1 и ГНСС2 частично отражает шум приёмников, разные положения антенн и неучтённое плечо до камеры.
- Если GGA quality не равен 4/5, нельзя называть референс сантиметровым RTK или делать отдельный вывод о качестве на RTK-участках.
- Временная калибровка `fx/fy`, упрощённые VO без полноценной оптимизации и монокулярный scale drift ограничивают выводы. Для производственного сравнения нужны калибровка камеры, внешние параметры камера–антенна и исходные ORB-SLAM2/DSO.